# Supplementary Multidimensional Similarity Analysis Notebook

This notebook is built for **Google Colab** and generates:

- **Table 7:** Advertisement Personality Profile Matrix (5D vectors)
- **Table 8:** Similarity Metrics Comparison (Euclidean vs Cosine)
- **Table 9:** Cross-Group Validation
- Appendix visuals:
  - 3D projection of trait vectors
  - Radar chart (multidimensional spillover)
  - Cosine angle visualization

> Replace file paths/column names only if your export differs.


## 1) Setup and imports


In [ ]:
import io
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import statsmodels.api as sm

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 2) Load your data

In Colab, run this cell and upload your CSV/TSV export when prompted.


In [ ]:
from google.colab import files
uploaded = files.upload()  # choose your data file

filename = list(uploaded.keys())[0]
print("Loaded:", filename)

# Tries CSV first, then TSV fallback
try:
    df = pd.read_csv(io.BytesIO(uploaded[filename]), low_memory=False)
except Exception:
    df = pd.read_csv(io.BytesIO(uploaded[filename]), sep="	", low_memory=False)

print("Shape:", df.shape)
df.head(2)


## 3) Cleaning and variable mapping

This section handles `#NULL!`, converts key columns to numeric, and defines analysis variables.


In [ ]:
# Standard null cleanup
null_tokens = ["#NULL!", "", " ", "NA", "N/A", "nan", "NaN", "None"]
df = df.replace(null_tokens, np.nan)

# Expected columns based on your schema
trait_cols = ["OPEN_Score", "CONSC_Score", "EXTRA_Score", "AGREE_Score", "NEURO_Score"]
click_items = ["CLIC1", "CLIC2", "CLIC3", "CLIC4"]
pers_items = ["PERS1", "PERS2", "PERS3", "PERS4", "PERS5", "PERS6"]
rele_items = ["RELE1", "RELE2", "RELE3"]

for c in trait_cols + click_items + pers_items + rele_items:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Composite outcomes (safe if some items missing)
avail_click = [c for c in click_items if c in df.columns]
avail_pers = [c for c in pers_items if c in df.columns]
avail_rele = [c for c in rele_items if c in df.columns]

if avail_click:
    df["CLICK_MEAN"] = df[avail_click].mean(axis=1)
if avail_pers:
    df["PERSUASIVE_MEAN"] = df[avail_pers].mean(axis=1)
if avail_rele:
    df["RELEVANCE_MEAN"] = df[avail_rele].mean(axis=1)

# Keep only rows with ad condition + trait scores
required = ["AdCond"] + trait_cols
analysis_df = df.dropna(subset=[c for c in required if c in df.columns]).copy()
print("Analysis rows:", len(analysis_df))
analysis_df[["AdCond"] + trait_cols].head()


## 4) Build ad-level 5D personality vectors (Table 7)

Ad vectors are estimated as mean participant trait profiles by `AdCond`.
If you already have externally estimated vectors, replace this table directly.


In [ ]:
ad_vector = (
    analysis_df.groupby("AdCond", dropna=True)[trait_cols]
    .mean()
    .rename(columns={
        "OPEN_Score": "Openness",
        "CONSC_Score": "Conscientiousness",
        "EXTRA_Score": "Extraversion",
        "AGREE_Score": "Agreeableness",
        "NEURO_Score": "Neuroticism",
    })
    .sort_index()
)

table7 = ad_vector.round(3)
print("Table 7: Advertisement Personality Profile Matrix (5D)")
display(table7)


## 5) Compute Euclidean and Cosine alignment per participant

For each respondent, compare their 5D trait vector to the 5D vector of the ad condition they saw.


In [ ]:
# Numeric matrix for calculations
ad_vectors_np = ad_vector[["OPEN_Score","CONSC_Score","EXTRA_Score","AGREE_Score","NEURO_Score"]].to_dict("index")

def euclidean(a, b):
    return float(np.linalg.norm(np.array(a) - np.array(b)))

def cosine(a, b):
    a = np.array(a).reshape(1, -1)
    b = np.array(b).reshape(1, -1)
    return float(cosine_similarity(a, b)[0, 0])

def row_similarity(row):
    ad = row["AdCond"]
    if ad not in ad_vectors_np:
        return pd.Series({"euclidean": np.nan, "cosine": np.nan})
    person = [row[c] for c in trait_cols]
    target = [ad_vectors_np[ad][c] for c in trait_cols]
    return pd.Series({
        "euclidean": euclidean(person, target),
        "cosine": cosine(person, target)
    })

analysis_df[["euclidean", "cosine"]] = analysis_df.apply(row_similarity, axis=1)
analysis_df[["AdCond", "euclidean", "cosine"]].head()


## 6) Table 8: Similarity Metrics Comparison

Shows ad-level means and the correlation between cosine similarity and click behavior.


In [ ]:
def safe_corr(x, y):
    tmp = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(tmp) < 3:
        return np.nan
    return tmp["x"].corr(tmp["y"])

table8 = analysis_df.groupby("AdCond").agg(
    Euclidean_Distance_Mean=("euclidean", "mean"),
    Cosine_Similarity_Mean=("cosine", "mean"),
    Corr_Cosine_Click=("cosine", lambda s: safe_corr(s, analysis_df.loc[s.index, "CLICK_MEAN"]) if "CLICK_MEAN" in analysis_df.columns else np.nan)
).round(3)

print("Table 8: Similarity Metrics Comparison")
display(table8)


## 7) Table 9: Cross-group validation

If `connectId` is available, we split by median `connectId` (or random fallback) into two groups and compare ad-level cosine means.


In [ ]:
temp = analysis_df.copy()

if "connectId" in temp.columns:
    # Try numeric split; fallback to deterministic hash-based split
    cid_num = pd.to_numeric(temp["connectId"], errors="coerce")
    if cid_num.notna().sum() > 0:
        cutoff = cid_num.median()
        temp["grp"] = np.where(cid_num <= cutoff, "Group 1", "Group 2")
    else:
        temp["grp"] = np.where(temp["connectId"].astype(str).str.len() % 2 == 0, "Group 1", "Group 2")
else:
    # Reproducible random split if no connectId
    rng = np.random.default_rng(42)
    temp["grp"] = np.where(rng.random(len(temp)) < 0.5, "Group 1", "Group 2")

pivot = temp.pivot_table(index="AdCond", columns="grp", values="cosine", aggfunc="mean")

if {"Group 1", "Group 2"}.issubset(pivot.columns):
    r_groups = pivot[["Group 1", "Group 2"]].corr().iloc[0,1]
else:
    r_groups = np.nan

table9 = pivot.rename(columns={"Group 1": "Cosine_Group1", "Group 2": "Cosine_Group2"}).round(3)
table9["r_between_groups"] = r_groups.round(3) if pd.notna(r_groups) else np.nan

print("Table 9: Cross-Group Validation")
display(table9)


## 8) Regression comparison

- Model A: `CLICK_MEAN ~ euclidean`
- Model B: `CLICK_MEAN ~ cosine`


In [ ]:
if "CLICK_MEAN" in analysis_df.columns:
    reg = analysis_df[["CLICK_MEAN", "euclidean", "cosine"]].dropna().copy()

    X_euc = sm.add_constant(reg[["euclidean"]])
    y = reg["CLICK_MEAN"]
    m1 = sm.OLS(y, X_euc).fit()

    X_cos = sm.add_constant(reg[["cosine"]])
    m2 = sm.OLS(y, X_cos).fit()

    print("Model A: CLICK_MEAN ~ euclidean")
    print(m1.summary())
    print("\nModel B: CLICK_MEAN ~ cosine")
    print(m2.summary())
else:
    print("CLICK_MEAN not available. Ensure CLIC1..CLIC4 are present.")


## 9) Appendix figures


In [ ]:
# 9.1 3D PCA projection of ad vectors
pca = PCA(n_components=3)
vec_5d = ad_vector[["OPEN_Score","CONSC_Score","EXTRA_Score","AGREE_Score","NEURO_Score"]].values
vec_3d = pca.fit_transform(vec_5d)

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(vec_3d[:,0], vec_3d[:,1], vec_3d[:,2], s=80)
for i, name in enumerate(ad_vector.index):
    ax.text(vec_3d[i,0], vec_3d[i,1], vec_3d[i,2], name)
ax.set_title("3D Projection of Advertisement 5D Trait Vectors (PCA)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
plt.show()


In [ ]:
# 9.2 Radar chart for multidimensional spillover
labels = ["Open", "Consc", "Extra", "Agree", "Neuro"]
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False)
angles = np.concatenate([angles, [angles[0]]])

fig, ax = plt.subplots(figsize=(8,8), subplot_kw={"polar": True})
for ad, row in ad_vector.iterrows():
    vals = [row["OPEN_Score"], row["CONSC_Score"], row["EXTRA_Score"], row["AGREE_Score"], row["NEURO_Score"]]
    vals = np.concatenate([vals, [vals[0]]])
    ax.plot(angles, vals, linewidth=2, label=ad)
    ax.fill(angles, vals, alpha=0.05)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)
ax.set_title("Radar Chart: Multidimensional Spillover Across Ad Conditions")
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.10))
plt.show()


In [ ]:
# 9.3 Cosine angle visualization for one selected ad and one participant
example = analysis_df.dropna(subset=trait_cols + ["AdCond"]).iloc[0]
ad_name = example["AdCond"]
p = np.array([example[c] for c in trait_cols], dtype=float)
a = np.array([ad_vectors_np[ad_name][c] for c in trait_cols], dtype=float)

cos_val = cosine_similarity(p.reshape(1,-1), a.reshape(1,-1))[0,0]
angle_deg = np.degrees(np.arccos(np.clip(cos_val, -1, 1)))

plt.figure(figsize=(6,4))
plt.bar(["Cosine Similarity"], [cos_val])
plt.ylim(-1, 1)
plt.title(f"Cosine Alignment for Example Case ({ad_name})\nAngle = {angle_deg:.1f}°")
plt.show()

print(f"Cosine similarity: {cos_val:.3f}")
print(f"Angle (degrees): {angle_deg:.2f}")


## 10) Export final tables to CSV


In [ ]:
out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)

table7.to_csv(f"{out_dir}/table7_ad_profile_matrix.csv")
table8.to_csv(f"{out_dir}/table8_similarity_comparison.csv")
table9.to_csv(f"{out_dir}/table9_cross_group_validation.csv")

print("Saved files:")
for f in os.listdir(out_dir):
    print("-", os.path.join(out_dir, f))

# Optional: download in Colab
from google.colab import files
for f in ["table7_ad_profile_matrix.csv", "table8_similarity_comparison.csv", "table9_cross_group_validation.csv"]:
    files.download(os.path.join(out_dir, f))
